# EDA and Baseline Model

PRD Chapters 3, 12, 13 — Mobile Price-Range Classifier.

Load cleaned real Kaggle dataset (`data/processed/train_cleaned.csv` or `merged_dataset.csv`),
validate missingness and domain zero-value imputations, build a scikit-learn preprocessing + RandomForest **baseline (no RFE)**,
run the nested CV schema (5 outer StratifiedKFold folds, inner hyperparameter search,
`random_state=42`), and report Accuracy + macro F1 on the held-out 20% stratified test split.



In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "backend"))

from app.services.inference import (
    FEATURE_COLUMNS,
    RANDOM_STATE,
    TARGET_COLUMN,
    build_baseline_pipeline,
    default_merged_csv_path,
    load_training_frame,
    save_model,
    validate_no_missing_features,
)

print("repo:", REPO_ROOT)
print("features:", len(FEATURE_COLUMNS))

repo: C:\Users\chaud\DesktopOG\Mobile_Price_Range_Classifier
features: 20


## 1. Load & filter training rows

Loads cleaned dataset (2,000 rows, 20 features, 4 classes).


In [2]:
raw = pd.read_csv(default_merged_csv_path())
print("merged shape:", raw.shape)
if "source" in raw.columns:
    print(raw["source"].value_counts())

df = load_training_frame()
print("training shape:", df.shape)
print("price_range balance:")
print(df[TARGET_COLUMN].value_counts().sort_index())
df.head()

merged shape: (2043, 23)
source
kaggle_train    2000
Name: count, dtype: int64
training shape: (2000, 21)
price_range balance:
price_range
0    500
1    500
2    500
3    500
Name: count, dtype: int64


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


## 2. Validate missing values & domain zero cleaning

In [3]:
validate_no_missing_features(df)
print("OK: no missing values in FEATURE_COLUMNS or price_range")
print(f"sc_w min: {df['sc_w'].min()} cm (zeros remaining: {(df['sc_w'] == 0).sum()})")
print(f"px_height min: {df['px_height'].min()} px (zeros remaining: {(df['px_height'] == 0).sum()})")

OK: no missing values in FEATURE_COLUMNS or price_range
sc_w min: 1 cm (zeros remaining: 0)
px_height min: 1 px (zeros remaining: 0)


## 3. 80/20 stratified split (seed = 42)



In [4]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"train={len(X_train)} test={len(X_test)}")
print("train class counts:\n", y_train.value_counts().sort_index())
print("test class counts:\n", y_test.value_counts().sort_index())

train=1600 test=400
train class counts:
 price_range
0    400
1    400
2    400
3    400
Name: count, dtype: int64
test class counts:
 price_range
0    100
1    100
2    100
3    100
Name: count, dtype: int64


## 4. Preprocessing + baseline pipeline



In [5]:
pipe = build_baseline_pipeline()
pipe

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('scale', StandardScaler(),
                                                  ['battery_power', 'blue',
                                                   'clock_speed', 'dual_sim',
                                                   'fc', 'four_g', 'int_memory',
                                                   'm_dep', 'mobile_wt',
                                                   'n_cores', 'pc', 'px_height',
                                                   'px_width', 'ram', 'sc_h',
                                                   'sc_w', 'talk_time',
                                                   'three_g', 'touch_screen',
                                                   'wifi'])])),
                ('clf',
                 RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                        random_state=42))])

## 5. Nested cross-validation schema (PRD Ch. 12)

- **Outer loop:** `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` — unbiased performance estimate
- **Inner loop:** `GridSearchCV` over RandomForest hyperparameters (baseline = all features)
- Scoring: `f1_macro`



In [6]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_leaf": [1, 2],
}

inner_search = GridSearchCV(
    estimator=build_baseline_pipeline(),
    param_grid=param_grid,
    cv=inner_cv,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True,
)

nested_scores = cross_val_score(
    inner_search, X_train, y_train, cv=outer_cv, scoring="f1_macro", n_jobs=-1
)
print("Nested CV macro F1 per outer fold:", np.round(nested_scores, 4))
print(
    f"Nested CV macro F1 mean+/-std: {nested_scores.mean():.4f} +/- {nested_scores.std():.4f}"
)

Nested CV macro F1 per outer fold: [0.8652 0.8692 0.8103 0.8889 0.897 ]
Nested CV macro F1 mean+/-std: 0.8661 +/- 0.0303


## 6. Fit on full train, evaluate once on held-out 20%

In [7]:
inner_search.fit(X_train, y_train)
best_model = inner_search.best_estimator_
print("Best params:", inner_search.best_params_)

y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Hold-out Accuracy:  {accuracy:.4f}")
print(f"Hold-out macro F1:  {macro_f1:.4f}")
print()
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Best params: {'clf__max_depth': 10, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 200}
Hold-out Accuracy:  0.8750
Hold-out macro F1:  0.8745

              precision    recall  f1-score   support

           0     0.9412    0.9600    0.9505       100
           1     0.8182    0.8100    0.8141       100
           2     0.8061    0.7900    0.7980       100
           3     0.9307    0.9400    0.9353       100

    accuracy                         0.8750       400
   macro avg     0.8740    0.8750    0.8745       400
weighted avg     0.8740    0.8750    0.8745       400

Confusion matrix:
 [[96  4  0  0]
 [ 6 81 13  0]
 [ 0 14 79  7]
 [ 0  0  6 94]]


## 7. Persist artifact + metrics JSON

In [8]:
artifact_path = save_model(best_model)
metrics = {
    "model": "baseline_random_forest_no_rfe",
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": len(FEATURE_COLUMNS),
    "best_params": inner_search.best_params_,
    "nested_cv_macro_f1_mean": float(nested_scores.mean()),
    "nested_cv_macro_f1_std": float(nested_scores.std()),
    "nested_cv_macro_f1_folds": [float(s) for s in nested_scores],
    "holdout_accuracy": float(accuracy),
    "holdout_macro_f1": float(macro_f1),
    "artifact_path": str(artifact_path.relative_to(REPO_ROOT)),
    "random_state": RANDOM_STATE,
}
metrics_path = REPO_ROOT / "artifacts" / "baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Saved", artifact_path)
print("Saved", metrics_path)
metrics

Saved C:\Users\chaud\DesktopOG\Mobile_Price_Range_Classifier\artifacts\baseline_rf.joblib
Saved C:\Users\chaud\DesktopOG\Mobile_Price_Range_Classifier\artifacts\baseline_metrics.json


{'model': 'baseline_random_forest_no_rfe',
 'n_train': 1600,
 'n_test': 400,
 'n_features': 20,
 'best_params': {'clf__max_depth': 10,
  'clf__min_samples_leaf': 2,
  'clf__n_estimators': 200},
 'nested_cv_macro_f1_mean': 0.8661188011977309,
 'nested_cv_macro_f1_std': 0.03033820697461455,
 'nested_cv_macro_f1_folds': [0.8651850939549725,
  0.8691812673664712,
  0.810286464076214,
  0.8889396797302425,
  0.8970015008607547],
 'holdout_accuracy': 0.875,
 'holdout_macro_f1': 0.8744671455820299,
 'artifact_path': 'artifacts\\baseline_rf.joblib',
 'random_state': 42}

## 8. RFECV Feature Selection (PRD Ch. 12.2)

Per PRD Chapter 12.2:
- Recursive Feature Elimination with 5-fold Stratified Cross-Validation (`RFECV`).
- Metric: `f1_macro`, step = 1, `min_features_to_select` = 5.
- Eliminates uninformative hardware specs to reduce complexity and avoid overfitting.

In [ ]:
from sklearn.feature_selection import RFECV

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_COLUMNS)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_COLUMNS)

base_rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rfecv = RFECV(
    estimator=base_rf,
    step=1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1_macro",
    min_features_to_select=5,
    n_jobs=-1,
)
rfecv.fit(X_train_scaled, y_train)

selected_features = [f for f, s in zip(FEATURE_COLUMNS, rfecv.support_) if s]
print(f"Optimal features count: {rfecv.n_features_}")
print(f"Selected features ({len(selected_features)}): {selected_features}")
print("\nFeature Rankings:")
for feat, rank in sorted(zip(FEATURE_COLUMNS, rfecv.ranking_), key=lambda x: x[1]):
    status = "SELECTED" if rank == 1 else f"rank {rank}"
    print(f"  {feat:15s}: {status}")

## 9. Hyperparameter Search on Selected Features (PRD Ch. 12.2)

Grid search over:
- `n_estimators`: [100, 200, 400]
- `max_depth`: [None, 10, 20]
- `min_samples_leaf`: [1, 2, 4]

In [ ]:
from sklearn.model_selection import cross_validate

param_grid = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 4],
}

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid,
    cv=inner_cv,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True,
)
grid.fit(X_train_scaled[selected_features], y_train)

print(f"Best Hyperparameters: {grid.best_params_}")
print(f"Best CV Macro F1: {grid.best_score_:.4f}")

# Cross-validate best estimator
cv_eval = cross_validate(
    grid.best_estimator_,
    X_train_scaled[selected_features],
    y_train,
    cv=inner_cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=-1,
)
cv_acc_mean = float(np.mean(cv_eval["test_accuracy"]))
cv_f1_mean = float(np.mean(cv_eval["test_f1_macro"]))
print(f"5-Fold CV Accuracy: {cv_acc_mean:.4f} (+/- {np.std(cv_eval['test_accuracy']):.4f})")
print(f"5-Fold CV Macro F1: {cv_f1_mean:.4f} (+/- {np.std(cv_eval['test_f1_macro']):.4f})")

## 10. Evaluate on Untouched 20% Hold-out Split (PRD Ch. 3)

Committed Targets:
- Hold-out Accuracy $\ge 0.90$ (90%)
- Hold-out Macro F1 $\ge 0.88$ (0.88)
- Materially fewer features than Month 1 baseline (5 vs 20 features)

In [ ]:
best_rf = grid.best_estimator_
y_pred_tuned = best_rf.predict(X_test_scaled[selected_features])

tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned, average="macro")

print("=" * 50)
print(f"Hold-out Test Accuracy: {tuned_acc:.4f} (Target: >= 0.9000)")
print(f"Hold-out Test Macro F1: {tuned_f1:.4f} (Target: >= 0.8800)")
print("=" * 50)

assert tuned_acc >= 0.90, f"Accuracy {tuned_acc:.4f} below target 0.90"
assert tuned_f1 >= 0.88, f"Macro F1 {tuned_f1:.4f} below target 0.88"
print("SUCCESS: Tuned model meets and exceeds committed targets on untouched test set!")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

## 11. Production Pipeline Serialization & Model Registry Entry

Construct an end-to-end `Pipeline` containing `ColumnTransformer` (subselecting and scaling the 5 features) and the tuned `RandomForestClassifier`.
Register the active model entry in `model_registry`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from app.db.models import ModelRegistry
from app.db.session import SessionLocal, init_db

# Build production pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), selected_features),
    ],
    remainder="drop",
)
preprocessor.fit(X_train)

tuned_rf = RandomForestClassifier(
    **grid.best_params_,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
X_train_preprocessed = preprocessor.transform(X_train)
tuned_rf.fit(X_train_preprocessed, y_train)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", tuned_rf),
])

# Verify pipeline predictions match
pipe_preds = pipeline.predict(X_test)
assert np.array_equal(pipe_preds, y_pred_tuned), "Pipeline output mismatch"

# Persist artifact
artifact_path = REPO_ROOT / "artifacts" / "tuned_rfecv_rf.joblib"
save_model(pipeline, artifact_path)
print("Saved tuned artifact:", artifact_path)

# Register in database model_registry
init_db()
db = SessionLocal()
try:
    db.query(ModelRegistry).filter(ModelRegistry.is_active.is_(True)).update({"is_active": False})
    
    version_tag = "v2.0.0-rfecv"
    existing = db.query(ModelRegistry).filter_by(version_tag=version_tag).first()
    if existing:
        existing.cv_accuracy = cv_acc_mean
        existing.cv_macro_f1 = cv_f1_mean
        existing.artifact_path = "artifacts/tuned_rfecv_rf.joblib"
        existing.is_active = True
    else:
        reg = ModelRegistry(
            version_tag=version_tag,
            cv_accuracy=cv_acc_mean,
            cv_macro_f1=cv_f1_mean,
            artifact_path="artifacts/tuned_rfecv_rf.joblib",
            is_active=True,
        )
        db.add(reg)
    db.commit()
    active = db.query(ModelRegistry).filter_by(is_active=True).first()
    print(f"Active model in registry: id={active.id}, tag={active.version_tag}, path={active.artifact_path}")
finally:
    db.close()